In [ ]:
library(dplyr)

#-- Load outcomes
outcome <- read.csv("/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Main_CYP_Outcomes_90days_CypModulators_2scripts_S_C.csv") %>%
  mutate(
    FirstDispense.treatment = as.Date(FirstDispense.treatment),
    ExpectedEndDate.treatment = as.Date(ExpectedEndDate.treatment),
    FirstDispense.treatment.episode = as.Date(FirstDispense.treatment.episode),
    ExpectedEndDate.treatment.episode = as.Date(ExpectedEndDate.treatment.episode),
    FirstDispense.other.episode = as.Date(FirstDispense.other.episode),
    ExpectedEndDate.other.episode = as.Date(ExpectedEndDate.other.episode),
    Incidence = as.Date(Incidence),
    DateFirstDispense = as.Date(DateFirstDispense)
  )

In [ ]:
library(readr)

workspace_bucket <- Sys.getenv("WORKSPACE_BUCKET")

cyp2c19_file_path <- file.path(workspace_bucket, "pgx/cyp2c19_cohort_results.tsv")
cyp2c19_df <- read_tsv(pipe(sprintf("gsutil cat %s", cyp2c19_file_path)))

cyp2b6_file_path <- file.path(workspace_bucket, "pgx/cyp2b6_cohort_results.tsv")
cyp2b6_df <- read_tsv(pipe(sprintf("gsutil cat %s", cyp2b6_file_path)))

cyp2d6_file_path <- file.path(workspace_bucket, "pgx/cyp2d6_cohort_results.tsv")
cyp2d6_df <- read_tsv(pipe(sprintf("gsutil cat %s", cyp2d6_file_path)))

In [ ]:
cyp2b6_df <- cyp2b6_df %>%
mutate(
CYP2B6 = case_when(
phenotype == "intermediate_metabolizer" ~ "Intermediate",
phenotype == "normal_metabolizer" ~ "Normal",
phenotype == "ultrarapid_metabolizer" ~ "Ultrarapid",
phenotype == "poor_metabolizer" ~ "Poor",
TRUE ~ NA_character_
)) %>%
select(person_id, CYP2B6, activity_score) %>%
rename(CYP2B6_ac = activity_score)
table(cyp2b6_df$CYP2B6, useNA="always")

In [ ]:
cyp2c19_df <- cyp2c19_df %>%
mutate(
CYP2C19 = case_when(
phenotype %in% c("Likely Intermediate Metabolizer", "Intermediate Metabolizer") ~ "Intermediate",
phenotype %in% c("Likely Poor Metabolizer", "Poor Metabolizer") ~ "Poor",
phenotype == "Normal Metabolizer" ~ "Normal",
phenotype == "Rapid Metabolizer" ~ "Rapid",
phenotype == "Ultrarapid Metabolizer" ~ "Ultrarapid",
TRUE ~ NA_character_)) %>%
select(person_id, CYP2C19)
table(cyp2c19_df$CYP2C19, useNA="always")

In [ ]:
cyp2d6_df <- cyp2d6_df %>%
mutate(
CYP2D6 = case_when(
phenotype == "CYP2D6 Intermediate Metabolizer" ~ "Intermediate",
phenotype == "CYP2D6 Poor Metabolizer" ~ "Poor",
phenotype == "CYP2D6 Normal Metabolizer" ~ "Normal",
phenotype == "CYP2D6 Ultrarapid Metabolizer" ~ "Ultrarapid",
TRUE ~ NA_character_)) %>%
mutate(CYP2D6_ac = as.numeric(gsub(">= ", "", gsub("CYP2D6 Activity Score ", "", activity_score)))) %>%
select(person_id, CYP2D6, CYP2D6_ac)
table(cyp2d6_df$CYP2D6, useNA="always")


In [ ]:
cyp <- full_join(full_join(cyp2b6_df, cyp2c19_df, by = "person_id"), cyp2d6_df, by = "person_id")
df <- left_join(outcome, cyp, by = "person_id")
total <- length(unique(df$person_id))
df_clean <- df %>% filter(!is.na(CYP2C19) & !is.na(CYP2B6) & !is.na(CYP2D6))
nonNA <- length(unique(df_clean$person_id))
total-nonNA

In [ ]:
cyp2c19_order <- c("Poor", "Intermediate", "Normal", "Rapid", "Ultrarapid")

apply_CYP2C19_ac_phenoconversion <- function(df,
                                             window = c("any_followup",
                                                        "baseline_only",
                                                        "concom_only")) {

  window <- match.arg(window)

  # build the relevant column names for this window
  any_inhib      <- paste0(window, "_cyp2c19_inhibitors")
  any_inhib_noAD <- paste0(window, "_cyp2c19_inhibitors_noAD")
  any_inhib_AD   <- paste0(window, "_cyp2c19_inhibitors_AD")
  s_inhib      <- paste0(window, "_cyp2c19_strong_inhibitors")
  s_inhib_noAD <- paste0(window, "_cyp2c19_strong_inhibitors_noAD")
  s_inhib_AD <- paste0(window, "_cyp2c19_strong_inhibitors_AD")
  m_inhib      <- paste0(window, "_cyp2c19_moderate_inhibitors")
  m_inhib_noAD <- paste0(window, "_cyp2c19_moderate_inhibitors_noAD")
  m_inhib_AD <- paste0(window, "_cyp2c19_moderate_inhibitors_AD")
  w_inhib      <- paste0(window, "_cyp2c19_weak_inhibitors")

  any_ind      <- paste0(window, "_cyp2c19_inducers")
  m_ind        <- paste0(window, "_cyp2c19_moderate_inducers")
  w_ind        <- paste0(window, "_cyp2c19_weak_inducers")

  new_c19 <- paste0("CYP2C19_", window, "_phenoconverted")

  df %>%
    mutate(
      ## modulation flags
      has_strong_inhib = .data[[s_inhib]] == 1L |
         .data[[s_inhib_noAD]] == 1L |
         .data[[s_inhib_AD]] == 1L,
      has_mod_inhib    = .data[[m_inhib]] == 1L |
         .data[[m_inhib_noAD]] == 1L |
         .data[[m_inhib_AD]] == 1L,
      has_weak_inhib   = .data[[w_inhib]] == 1L,
      has_any_inhib    = has_strong_inhib | has_mod_inhib | has_weak_inhib,

      ## net delta with inhibitor dominance
      cyp2c19_delta = case_when(
        has_strong_inhib ~ -99L,
        !has_strong_inhib & has_mod_inhib ~ -1L,
        !has_strong_inhib & !has_mod_inhib & has_weak_inhib ~ -1L,
        !has_any_inhib & .data[[any_ind]] == 1L ~ 2L,
        !has_any_inhib & .data[[m_ind]]   == 1L ~ 1L,
        !has_any_inhib & .data[[w_ind]]   == 1L ~ 1L,
        TRUE ~ 0L
      ),

      ## phenotype index
      cyp2c19_index = match(CYP2C19, cyp2c19_order),
      cyp2c19_index_conv = pmin(pmax(cyp2c19_index + cyp2c19_delta, 1L),
                                length(cyp2c19_order)),

      !!new_c19 := ifelse(
        is.na(cyp2c19_index),
        NA_character_,
        cyp2c19_order[cyp2c19_index_conv]
      )
    )
}


In [ ]:
cyp_order <- c(
  "Poor",
  "Intermediate",
  "Normal",
  "Ultrarapid"
)

phenotype_from_AS <- function(as) {
  case_when(
    is.na(as)          ~ NA_character_,
    as == 0            ~ cyp_order[1],          # PM
    as > 0   & as <= 1 ~ cyp_order[2],          # IM
    as > 1   & as <= 2 ~ cyp_order[3],          # NM
    as > 2             ~ cyp_order[4]           # UM
  )
}

apply_CYP2D6_B6_ac_phenoconversion <- function(df,
                                               window = c("any_followup",
                                                          "baseline_only",
                                                          "concom_only")) {
  window <- match.arg(window)

  ## ----- build window-specific column names -----

  # CYP2D6 inhibitors (with and without antidepressants)
  d6_any_inhib      <- paste0(window, "_cyp2d6_inhibitors")
  d6_any_inhib_noAD <- paste0(window, "_cyp2d6_inhibitors_noAD")
  d6_any_inhib_AD   <- paste0(window, "_cyp2d6_inhibitors_AD")
  d6_s_inhib        <- paste0(window, "_cyp2d6_strong_inhibitors")
  d6_s_inhib_noAD   <- paste0(window, "_cyp2d6_strong_inhibitors_noAD")
  d6_s_inhib_AD     <- paste0(window, "_cyp2d6_strong_inhibitors_AD")
  d6_m_inhib        <- paste0(window, "_cyp2d6_moderate_inhibitors")
  d6_m_inhib_noAD   <- paste0(window, "_cyp2d6_moderate_inhibitors_noAD")
  d6_m_inhib_AD     <- paste0(window, "_cyp2d6_moderate_inhibitors_AD")
  d6_w_inhib        <- paste0(window, "_cyp2d6_weak_inhibitors")
  d6_w_inhib_noAD   <- paste0(window, "_cyp2d6_weak_inhibitors_noAD")
  d6_w_inhib_AD     <- paste0(window, "_cyp2d6_weak_inhibitors_AD")

  # CYP2B6 inhibitors
  b6_any_inhib <- paste0(window, "_cyp2b6_inhibitors")
  b6_s_inhib   <- paste0(window, "_cyp2b6_strong_inhibitors")
  b6_m_inhib   <- paste0(window, "_cyp2b6_moderate_inhibitors")
  b6_w_inhib   <- paste0(window, "_cyp2b6_weak_inhibitors")

  # CYP2B6 inducers
  b6_any_ind   <- paste0(window, "_cyp2b6_inducers")
  b6_m_ind     <- paste0(window, "_cyp2b6_moderate_inducers")
  b6_w_ind     <- paste0(window, "_cyp2b6_weak_inducers")

  # dynamic output names
  new_d6 <- paste0("CYP2D6_", window, "_phenoconverted")
  new_b6 <- paste0("CYP2B6_", window, "_phenoconverted")

  df %>%
    mutate(
      ## -----------------------------
      ## CYP2D6: inhibitors only
      ## -----------------------------
      d6_has_strong_inhib = .data[[d6_s_inhib]]      == 1L |
                            .data[[d6_s_inhib_noAD]] == 1L,

      d6_has_mod_or_weak_inhib = (
        .data[[d6_m_inhib]]      == 1L |
        .data[[d6_m_inhib_noAD]] == 1L |
        .data[[d6_w_inhib]]      == 1L |
        .data[[d6_w_inhib_noAD]] == 1L |
        .data[[d6_any_inhib]]    == 1L |
        .data[[d6_any_inhib_noAD]] == 1L
      ) & !d6_has_strong_inhib,

      CYP2D6_inhib_factor = case_when(
        is.na(CYP2D6_ac)         ~ NA_real_,
        d6_has_strong_inhib      ~ 0.0,  # strong inhibitor → PM-equivalent
        d6_has_mod_or_weak_inhib ~ 0.5,  # any other inhibitor → 50% activity
        TRUE                     ~ 1.0
      ),

      CYP2D6_ac_adj = CYP2D6_ac * CYP2D6_inhib_factor,

      ## -----------------------------
      ## CYP2B6: inhibitors + inducers
      ## -----------------------------
      b6_has_strong_inhib = .data[[b6_s_inhib]] == 1L,
      b6_has_mod_inhib    = .data[[b6_m_inhib]] == 1L |
                            .data[[b6_w_inhib]] == 1L,
      b6_has_any_inhib    = .data[[b6_any_inhib]] == 1L |
                            b6_has_strong_inhib | b6_has_mod_inhib,

      CYP2B6_inhib_factor = case_when(
        is.na(CYP2B6_ac)     ~ NA_real_,
        b6_has_strong_inhib  ~ 0.0,  # strong → PM-equivalent
        b6_has_mod_inhib     ~ 0.5,  # moderate/weak/"any" (without strong)
        TRUE                 ~ 1.0
      ),

      # inducers (treat "any_inducers" like a strong-ish inducer, parallel to your CYP2C19 pattern)
      CYP2B6_induc_factor = case_when(
        is.na(CYP2B6_ac)          ~ NA_real_,
        .data[[b6_any_ind]] == 1L ~ 2.0,   # "any" / strong-ish
        .data[[b6_m_ind]]   == 1L ~ 1.5,   # moderate
        .data[[b6_w_ind]]   == 1L ~ 1.25,  # weak
        TRUE                       ~ 1.0
      ),

      # inhibitor dominance: if any inhibitor present, ignore inducer
      CYP2B6_net_factor = case_when(
        is.na(CYP2B6_ac) ~ NA_real_,
        b6_has_any_inhib ~ CYP2B6_inhib_factor,
        TRUE             ~ CYP2B6_induc_factor
      ),

      CYP2B6_ac_adj = CYP2B6_ac * CYP2B6_net_factor,

      ## -----------------------------
      ## FINAL phenoconverted phenotypes (window-specific names)
      ## -----------------------------
      !!new_d6 := phenotype_from_AS(CYP2D6_ac_adj),
      !!new_b6 := phenotype_from_AS(CYP2B6_ac_adj)
    )
}


In [ ]:
#-- Apply phenoconversion to cyp2c19
cyp2c19_any_followup <- apply_CYP2C19_ac_phenoconversion(df, "any_followup")
cyp2c19_baseline_only <- apply_CYP2C19_ac_phenoconversion(df, "baseline_only")
cyp2c19_concom_only <- apply_CYP2C19_ac_phenoconversion(df, "concom_only")
dim(cyp2c19_any_followup)

#-- Apply phenoconversion to cyp2d6 and cyp2b6
cyp_any_followup <- apply_CYP2D6_B6_ac_phenoconversion(df, "any_followup")
cyp_baseline_only <- apply_CYP2D6_B6_ac_phenoconversion(df, "baseline_only")
cyp_concom_only <- apply_CYP2D6_B6_ac_phenoconversion(df, "concom_only")

dim(cyp_any_followup)

In [ ]:
#-- CREATE INDICATOR FOR AD INHIBITOR PHENOCONVERSION (instead of filtering)

# CYP2D6
cyp_any_followup <- cyp_any_followup %>%
  mutate(
    has_AD_cyp2d6_modulator_any_followup = case_when(
      any_followup_cyp2d6_inhibitors_AD == 1L ~ 1L,
      TRUE ~ 0L
    ),
    has_AD_cyp2d6_weak_inhibitors_any_followup = case_when(
        any_followup_cyp2d6_weak_inhibitors_AD == 1L ~ 1L,
        TRUE ~ 0L),
    has_non_AD_cyp2d6_weak_inhibitors_any_followup = case_when(
        any_followup_cyp2d6_weak_inhibitors_noAD == 1L ~ 1L,
        TRUE ~ 0L),
    has_AD_cyp2d6_moderate_inhibitors_any_followup = case_when(
        any_followup_cyp2d6_moderate_inhibitors_AD == 1L ~ 1L,
        TRUE ~ 0L),
    has_non_AD_cyp2d6_moderate_inhibitors_any_followup = case_when(
        any_followup_cyp2d6_moderate_inhibitors_noAD == 1L ~ 1L,
        TRUE ~ 0L),
    has_AD_cyp2d6_strong_inhibitors_any_followup = case_when(
        any_followup_cyp2d6_strong_inhibitors_AD == 1L ~ 1L,
        TRUE ~ 0L),
    has_non_AD_cyp2d6_strong_inhibitors_any_followup = case_when(
        any_followup_cyp2d6_strong_inhibitors_noAD == 1L ~ 1L,
        TRUE ~ 0L)
  )

cyp_baseline_only <- cyp_baseline_only %>%
  mutate(
    has_AD_cyp2d6_modulator_baseline_only = case_when(
      baseline_only_cyp2d6_inhibitors_AD == 1L ~ 1L,
      TRUE ~ 0L
    ),
    has_AD_cyp2d6_weak_inhibitors_baseline_only = case_when(
        baseline_only_cyp2d6_weak_inhibitors_AD == 1L ~ 1L,
        TRUE ~ 0L),
    has_non_AD_cyp2d6_weak_inhibitors_baseline_only = case_when(
        baseline_only_cyp2d6_weak_inhibitors_noAD == 1L ~ 1L,
        TRUE ~ 0L),
    has_AD_cyp2d6_moderate_inhibitors_baseline_only = case_when(
        baseline_only_cyp2d6_moderate_inhibitors_AD == 1L ~ 1L,
        TRUE ~ 0L),
    has_non_AD_cyp2d6_moderate_inhibitors_baseline_only = case_when(
        baseline_only_cyp2d6_moderate_inhibitors_noAD == 1L ~ 1L,
        TRUE ~ 0L),
    has_AD_cyp2d6_strong_inhibitors_baseline_only = case_when(
        baseline_only_cyp2d6_strong_inhibitors_AD == 1L ~ 1L,
        TRUE ~ 0L),
    has_non_AD_cyp2d6_strong_inhibitors_baseline_only = case_when(
        baseline_only_cyp2d6_strong_inhibitors_noAD == 1L ~ 1L,
        TRUE ~ 0L)
  )

cyp_concom_only <- cyp_concom_only %>%
  mutate(
    has_AD_cyp2d6_modulator_concom_only = case_when(
      concom_only_cyp2d6_inhibitors_AD == 1L ~ 1L,
      TRUE ~ 0L
    ),
    has_AD_cyp2d6_weak_inhibitors_concom_only = case_when(
        concom_only_cyp2d6_weak_inhibitors_AD == 1L ~ 1L,
        TRUE ~ 0L),
    has_non_AD_cyp2d6_weak_inhibitors_concom_only = case_when(
        concom_only_cyp2d6_weak_inhibitors_noAD == 1L ~ 1L,
        TRUE ~ 0L),
    has_AD_cyp2d6_moderate_inhibitors_concom_only = case_when(
        concom_only_cyp2d6_moderate_inhibitors_AD == 1L ~ 1L,
        TRUE ~ 0L),
    has_non_AD_cyp2d6_moderate_inhibitors_concom_only = case_when(
        concom_only_cyp2d6_moderate_inhibitors_noAD == 1L ~ 1L,
        TRUE ~ 0L),
    has_AD_cyp2d6_strong_inhibitors_concom_only = case_when(
        concom_only_cyp2d6_strong_inhibitors_AD == 1L ~ 1L,
        TRUE ~ 0L),
    has_non_AD_cyp2d6_strong_inhibitors_concom_only = case_when(
        concom_only_cyp2d6_strong_inhibitors_noAD == 1L ~ 1L,
        TRUE ~ 0L)
  )

# CYP2C19
cyp2c19_any_followup <- cyp2c19_any_followup %>%
  mutate(
    has_AD_cyp2c19_modulator_any_followup = case_when(
      any_followup_cyp2c19_inhibitors_AD == 1L ~ 1L,
      TRUE ~ 0L
    ),
    has_AD_cyp2c19_moderate_inhibitors_any_followup = case_when(
        any_followup_cyp2c19_moderate_inhibitors_AD == 1L ~ 1L,
        TRUE ~ 0L),
    has_non_AD_cyp2c19_moderate_inhibitors_any_followup = case_when(
        any_followup_cyp2c19_moderate_inhibitors_noAD == 1L ~ 1L,
        TRUE ~ 0L),
    has_AD_cyp2c19_strong_inhibitors_any_followup = case_when(
        any_followup_cyp2c19_strong_inhibitors_AD == 1L ~ 1L,
        TRUE ~ 0L),
    has_non_AD_cyp2c19_strong_inhibitors_any_followup = case_when(
        any_followup_cyp2c19_strong_inhibitors_noAD == 1L ~ 1L,
        TRUE ~ 0L)
  )

cyp2c19_baseline_only <- cyp2c19_baseline_only %>%
  mutate(
    has_AD_cyp2c19_modulator_baseline_only = case_when(
      baseline_only_cyp2c19_inhibitors_AD == 1L ~ 1L,
      TRUE ~ 0L
    ),
    has_AD_cyp2c19_moderate_inhibitors_baseline_only = case_when(
        baseline_only_cyp2c19_moderate_inhibitors_AD == 1L ~ 1L,
        TRUE ~ 0L),
    has_non_AD_cyp2c19_moderate_inhibitors_baseline_only = case_when(
        baseline_only_cyp2c19_moderate_inhibitors_noAD == 1L ~ 1L,
        TRUE ~ 0L),
    has_AD_cyp2c19_strong_inhibitors_baseline_only = case_when(
        baseline_only_cyp2c19_strong_inhibitors_AD == 1L ~ 1L,
        TRUE ~ 0L),
    has_non_AD_cyp2c19_strong_inhibitors_baseline_only = case_when(
        baseline_only_cyp2c19_strong_inhibitors_noAD == 1L ~ 1L,
        TRUE ~ 0L)
  )

cyp2c19_concom_only <- cyp2c19_concom_only %>%
  mutate(
    has_AD_cyp2c19_modulator_concom_only = case_when(
      concom_only_cyp2c19_inhibitors_AD == 1L ~ 1L,
      TRUE ~ 0L
    ),
    has_AD_cyp2c19_moderate_inhibitors_concom_only = case_when(
        concom_only_cyp2c19_moderate_inhibitors_AD == 1L ~ 1L,
        TRUE ~ 0L),
    has_non_AD_cyp2c19_moderate_inhibitors_concom_only = case_when(
        concom_only_cyp2c19_moderate_inhibitors_noAD == 1L ~ 1L,
        TRUE ~ 0L),
    has_AD_cyp2c19_strong_inhibitors_concom_only = case_when(
        concom_only_cyp2c19_strong_inhibitors_AD == 1L ~ 1L,
        TRUE ~ 0L),
    has_non_AD_cyp2c19_strong_inhibitors_concom_only = case_when(
        concom_only_cyp2c19_strong_inhibitors_noAD == 1L ~ 1L,
        TRUE ~ 0L)
  )

# Check counts
table(cyp_any_followup$has_AD_cyp2d6_modulator_any_followup)
table(cyp2c19_any_followup$has_AD_cyp2c19_modulator_any_followup)

In [ ]:
general <- df %>%
    select(person_id, drug.treatment, FirstDispense.treatment, ExpectedEndDate.treatment, FinalClassification.treatment,
          FirstDispense.treatment.episode, ExpectedEndDate.treatment.episode, EpisodeDuration.treatment.episode, drug.other,
          FirstDispense.other.episode, ExpectedEndDate.other.episode, EpisodeDuration.other.episode, Incidence,
          DateFirstDispense, General_adherence_score, ATC_Adherence_score, Category.treatment, Category.other, 
          General_adherence_score.prior, ATC_adherence_score.prior, Index, 
          concom_only_gabapentin_neg_cont, concom_only_levothyroxine_neg_cont, concom_only_pravastatin_neg_cont, concom_only_rosuvastatin_neg_cont,
          baseline_only_gabapentin_neg_cont, baseline_only_levothyroxine_neg_cont, baseline_only_pravastatin_neg_cont, baseline_only_rosuvastatin_neg_cont,
          any_followup_gabapentin_neg_cont, any_followup_levothyroxine_neg_cont, any_followup_pravastatin_neg_cont, any_followup_rosuvastatin_neg_cont)

In [ ]:
baseline_all <- cyp2c19_baseline_only %>%
  # CYP2C19 side
  select(
    person_id, drug.treatment,
    CYP2C19_baseline_only_phenoconverted,
    CYP2C19,
    has_AD_cyp2c19_modulator_baseline_only,
    has_AD_cyp2c19_moderate_inhibitors_baseline_only,
    has_AD_cyp2c19_strong_inhibitors_baseline_only,
    has_non_AD_cyp2c19_moderate_inhibitors_baseline_only,
    has_non_AD_cyp2c19_strong_inhibitors_baseline_only,
    baseline_only_cyp2c19_inducers,
    baseline_only_cyp2c19_inhibitors,
    baseline_only_cyp2c19_weak_inducers,
    baseline_only_cyp2c19_weak_inhibitors,
    baseline_only_cyp2c19_moderate_inhibitors,
    baseline_only_cyp2c19_moderate_inducers,
    baseline_only_cyp2c19_strong_inhibitors
  ) %>%
  # join CYP2D6 + CYP2B6 for the same window
  full_join(
    cyp_baseline_only %>%
      select(
        person_id, drug.treatment,
        # CYP2D6
        CYP2D6_baseline_only_phenoconverted,
        CYP2D6, CYP2D6_ac,
        has_AD_cyp2d6_modulator_baseline_only,
        has_AD_cyp2d6_weak_inhibitors_baseline_only,
        has_AD_cyp2d6_moderate_inhibitors_baseline_only,
        has_AD_cyp2d6_strong_inhibitors_baseline_only,
        has_non_AD_cyp2d6_weak_inhibitors_baseline_only,
        has_non_AD_cyp2d6_moderate_inhibitors_baseline_only,
        has_non_AD_cyp2d6_strong_inhibitors_baseline_only,
        baseline_only_cyp2d6_inhibitors,
        baseline_only_cyp2d6_weak_inhibitors,
        baseline_only_cyp2d6_moderate_inhibitors,
        baseline_only_cyp2d6_strong_inhibitors,
        # CYP2B6
        CYP2B6_baseline_only_phenoconverted,
        CYP2B6, CYP2B6_ac,
        baseline_only_cyp2b6_inhibitors,
        baseline_only_cyp2b6_weak_inhibitors,
        baseline_only_cyp2b6_moderate_inhibitors,
        baseline_only_cyp2b6_strong_inhibitors,
        baseline_only_cyp2b6_inducers,
        baseline_only_cyp2b6_weak_inducers,
        baseline_only_cyp2b6_moderate_inducers
      ),
    by = c("person_id", "drug.treatment")
  )


In [ ]:
any_followup_all <- cyp2c19_any_followup %>%
  select(
    person_id, drug.treatment,
    CYP2C19_any_followup_phenoconverted,
    has_AD_cyp2c19_modulator_any_followup,
    has_AD_cyp2c19_moderate_inhibitors_any_followup,
    has_AD_cyp2c19_strong_inhibitors_any_followup,
    has_non_AD_cyp2c19_moderate_inhibitors_any_followup,
    has_non_AD_cyp2c19_strong_inhibitors_any_followup,
    any_followup_cyp2c19_inducers,
    any_followup_cyp2c19_inhibitors,
    any_followup_cyp2c19_weak_inducers,
    any_followup_cyp2c19_weak_inhibitors,
    any_followup_cyp2c19_moderate_inhibitors,
    any_followup_cyp2c19_moderate_inducers,
    any_followup_cyp2c19_strong_inhibitors
  ) %>%
  full_join(
    cyp_any_followup %>%
      select(
        person_id, drug.treatment,
        CYP2D6_any_followup_phenoconverted,
        has_AD_cyp2d6_modulator_any_followup,
        has_AD_cyp2d6_weak_inhibitors_any_followup,
        has_AD_cyp2d6_moderate_inhibitors_any_followup,
        has_AD_cyp2d6_strong_inhibitors_any_followup,
        has_non_AD_cyp2d6_weak_inhibitors_any_followup,
        has_non_AD_cyp2d6_moderate_inhibitors_any_followup,
        has_non_AD_cyp2d6_strong_inhibitors_any_followup,
        any_followup_cyp2d6_inhibitors,
        any_followup_cyp2d6_weak_inhibitors,
        any_followup_cyp2d6_moderate_inhibitors,
        any_followup_cyp2d6_strong_inhibitors,
        CYP2B6_any_followup_phenoconverted,
        any_followup_cyp2b6_inhibitors,
        any_followup_cyp2b6_weak_inhibitors,
        any_followup_cyp2b6_moderate_inhibitors,
        any_followup_cyp2b6_strong_inhibitors,
        any_followup_cyp2b6_inducers,
        any_followup_cyp2b6_weak_inducers,
        any_followup_cyp2b6_moderate_inducers
      ),
    by = c("person_id", "drug.treatment")
  )


In [ ]:
concom_only_all <- cyp2c19_concom_only %>%
  select(
    person_id, drug.treatment,
    has_AD_cyp2c19_modulator_concom_only,
    has_AD_cyp2c19_moderate_inhibitors_concom_only,
    has_AD_cyp2c19_strong_inhibitors_concom_only,
    has_non_AD_cyp2c19_moderate_inhibitors_concom_only,
    has_non_AD_cyp2c19_strong_inhibitors_concom_only,
    CYP2C19_concom_only_phenoconverted,
    concom_only_cyp2c19_inducers,
    concom_only_cyp2c19_inhibitors,
    concom_only_cyp2c19_weak_inducers,
    concom_only_cyp2c19_weak_inhibitors,
    concom_only_cyp2c19_moderate_inhibitors,
    concom_only_cyp2c19_moderate_inducers,
    concom_only_cyp2c19_strong_inhibitors
  ) %>%
  full_join(
    cyp_concom_only %>%
      select(
        person_id, drug.treatment,
        CYP2D6_concom_only_phenoconverted,
        has_AD_cyp2d6_modulator_concom_only,
        has_AD_cyp2d6_weak_inhibitors_concom_only,
        has_AD_cyp2d6_moderate_inhibitors_concom_only,
        has_AD_cyp2d6_strong_inhibitors_concom_only,
        has_non_AD_cyp2d6_weak_inhibitors_concom_only,
        has_non_AD_cyp2d6_moderate_inhibitors_concom_only,
        has_non_AD_cyp2d6_strong_inhibitors_concom_only,
        concom_only_cyp2d6_inhibitors,
        concom_only_cyp2d6_weak_inhibitors,
        concom_only_cyp2d6_moderate_inhibitors,
        concom_only_cyp2d6_strong_inhibitors,
        CYP2B6_concom_only_phenoconverted,
        concom_only_cyp2b6_inhibitors,
        concom_only_cyp2b6_weak_inhibitors,
        concom_only_cyp2b6_moderate_inhibitors,
        concom_only_cyp2b6_strong_inhibitors,
        concom_only_cyp2b6_inducers,
        concom_only_cyp2b6_weak_inducers,
        concom_only_cyp2b6_moderate_inducers
      ),
    by = c("person_id", "drug.treatment")
  )


In [ ]:
all <- general %>%
    left_join(baseline_all, by = c("person_id", "drug.treatment")) %>%
    left_join(any_followup_all, by = c("person_id", "drug.treatment")) %>%
    left_join(concom_only_all,  by = c("person_id", "drug.treatment"))

In [ ]:
#-- Save results for first analyses. Participants can have more than one classification.
write.csv(all, "/home/jupyter/workspaces/geneticsofantidepressantresponse/data/Main_CYP_Outcomes_90days_Phenoconversion_noAD_2scripts_S_C.csv", row.names = FALSE, quote = FALSE)
